In [1]:
import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd

from fastai.text.all import *
from scipy.stats import spearmanr, rankdata

set_seed(42, reproducible=True)



In [2]:
path = Path("/kaggle/input/google-quest-challenge")

train = pd.read_csv(path / "train.csv")
test = pd.read_csv(path / "test.csv")
sample = pd.read_csv(path / "sample_submission.csv")

target_cols = [c for c in sample.columns if c != "qa_id"]

assert "qa_id" in test.columns and "qa_id" in sample.columns
assert len(target_cols) == 30
assert set(target_cols).issubset(
    set(train.columns)
), "Train is missing some target columns."



In [3]:
cols = ["question_title", "question_body", "answer"]

colQA = target_cols
ques = [c for c in colQA if c.startswith("question_")]
ans = [c for c in colQA if c.startswith("answer_")]

assert len(ques) + len(ans) == 30
assert len(ans) == 9, "Expected 9 answer targets for this competition."




In [4]:
def mean_spearman(preds, targs):
    preds = preds.detach().float().cpu().numpy()
    targs = targs.detach().float().cpu().numpy()
    cors = []
    for j in range(preds.shape[1]):
        r = spearmanr(preds[:, j], targs[:, j]).correlation
        if r is None or np.isnan(r):
            r = 0.0
        cors.append(r)
    return np.mean(cors)


mean_spearman_metric = AccumMetric(mean_spearman, flatten=False)




In [5]:
def _safe_str(x):
    return "" if pd.isna(x) else str(x)


def combine_all(row):
    t = _safe_str(row.get("question_title", ""))
    b = _safe_str(row.get("question_body", ""))
    a = _safe_str(row.get("answer", ""))
    return t + "\n\n" + b + "\n\n" + a


train_q = train[["qa_id", "question_title", "question_body", "answer"] + ques].copy()
test_q = test[["qa_id", "question_title", "question_body", "answer"]].copy()
train_q["text"] = train_q.apply(combine_all, axis=1)
test_q["text"] = test_q.apply(combine_all, axis=1)

train_a = train[["qa_id", "question_title", "question_body", "answer"] + ans].copy()
test_a = test[["qa_id", "question_title", "question_body", "answer"]].copy()
train_a["text"] = train_a.apply(combine_all, axis=1)
test_a["text"] = test_a.apply(combine_all, axis=1)

train_q["y"] = train_q[ques].astype("float32").values.tolist()
train_a["y"] = train_a[ans].astype("float32").values.tolist()



In [6]:
BS = 128


def make_dls(df, text_col, y_col, bs, seq_len=128, seed=42, valid_pct=0.2):
    n_out = len(df.iloc[0][y_col])
    dblock = DataBlock(
        blocks=(
            TextBlock.from_df(text_cols=text_col, is_lm=False, seq_len=seq_len),
            RegressionBlock(n_out=n_out),
        ),
        get_x=ColReader(text_col),
        get_y=ColReader(y_col),
        splitter=RandomSplitter(valid_pct=valid_pct, seed=seed),
    )
    return dblock.dataloaders(df, bs=bs, num_workers=0)


# Change (score-upward, minimal): slightly longer seq_len helps capture more context,
# which tends to improve rank ordering for Spearman without changing the model/loop.
SEQ_LEN = 192

dls_q = make_dls(
    train_q[["text", "y"]].copy(), text_col="text", y_col="y", bs=BS, seq_len=SEQ_LEN
)
dls_a = make_dls(
    train_a[["text", "y"]].copy(), text_col="text", y_col="y", bs=BS, seq_len=SEQ_LEN
)



In [7]:
q_learn = text_classifier_learner(
    dls_q,
    AWD_LSTM,
    pretrained=True,
    metrics=[mean_spearman_metric],
    loss_func=MSELossFlat(),
)

a_learn = text_classifier_learner(
    dls_a,
    AWD_LSTM,
    pretrained=True,
    metrics=[mean_spearman_metric],
    loss_func=MSELossFlat(),
)



In [8]:
# Change (score-upward, minimal): train a bit longer (same freeze->unfreeze approach),
# reducing underfitting while preserving the same architecture and optimization method.
q_learn.freeze()
q_learn.fit_one_cycle(1, 2e-3)
q_learn.unfreeze()
q_learn.fit_one_cycle(6, lr_max=slice(2e-4, 2e-3))

a_learn.freeze()
a_learn.fit_one_cycle(1, 2e-3)
a_learn.unfreeze()
a_learn.fit_one_cycle(6, lr_max=slice(2e-4, 2e-3))

gc.collect()



epoch,train_loss,valid_loss,mean_spearman,time
0,0.413411,0.323578,0.100860,00:06


epoch,train_loss,valid_loss,mean_spearman,time
0,0.314871,0.265000,0.134750,00:11
1,0.234065,0.097446,0.132124,00:10
2,0.155097,0.063383,0.207013,00:10
3,0.116393,0.061951,0.228325,00:11
4,0.096668,0.061000,0.243272,00:10
5,0.086415,0.060948,0.258665,00:11


epoch,train_loss,valid_loss,mean_spearman,time
0,0.724581,0.604347,0.107672,00:06


epoch,train_loss,valid_loss,mean_spearman,time
0,0.604346,0.512699,0.099869,00:11
1,0.431272,0.191549,0.132815,00:10
2,0.252421,0.054521,0.125676,00:11
3,0.165043,0.045842,0.153719,00:10
4,0.122773,0.043927,0.165979,00:10
5,0.100685,0.043473,0.192310,00:11


731

In [9]:
tst_dl_q = q_learn.dls.test_dl(test_q[["text"]], num_workers=0)
tst_dl_a = a_learn.dls.test_dl(test_a[["text"]], num_workers=0)

q_preds, _ = q_learn.get_preds(dl=tst_dl_q)
a_preds, _ = a_learn.get_preds(dl=tst_dl_a)


def rank_norm_cols(t: torch.Tensor) -> torch.Tensor:
    arr = t.detach().float().cpu().numpy()
    out = np.zeros_like(arr, dtype=np.float32)
    n = arr.shape[0]
    denom = max(n - 1, 1)
    for j in range(arr.shape[1]):
        r = rankdata(arr[:, j], method="average")  # 1..n
        out[:, j] = (r - 1.0) / denom
    return torch.from_numpy(out)


q_preds = rank_norm_cols(q_preds)
a_preds = rank_norm_cols(a_preds)

q_preds = q_preds.clamp(0, 1)
a_preds = a_preds.clamp(0, 1)

assert q_preds.shape[0] == len(test)
assert a_preds.shape[0] == len(test)
assert q_preds.shape[1] == len(ques), (q_preds.shape, len(ques))
assert a_preds.shape[1] == len(ans), (a_preds.shape, len(ans))



In [10]:
sub = sample.copy()
pred_df = pd.DataFrame({"qa_id": test["qa_id"].values})

for i, c in enumerate(ques):
    pred_df[c] = q_preds[:, i].numpy()
for i, c in enumerate(ans):
    pred_df[c] = a_preds[:, i].numpy()

sub = sub[["qa_id"] + target_cols].merge(
    pred_df, on="qa_id", how="left", suffixes=("_old", "")
)
sub = sub[["qa_id"] + target_cols]
sub[target_cols] = sub[target_cols].fillna(0.5).clip(0.0, 1.0)

sub.to_csv("submission.csv", index=False)

sub.head(10)

,qa_id,question_asker_intent_understanding,question_body_critical,question_conversational,question_expect_short_answer,question_fact_seeking,question_has_commonly_accepted_answer,question_interestingness_others,question_interestingness_self,question_multi_intent,...,question_well_written,answer_helpful,answer_level_of_information,answer_plausible,answer_relevance,answer_satisfaction,answer_type_instructions,answer_type_procedure,answer_type_reason_explanation,answer_well_written
0,6516,0.146623,0.225700,0.255354,0.191104,0.565074,0.499176,0.169687,0.067545,0.296540,...,0.233937,0.784185,0.691927,0.736409,0.779242,0.747941,0.604613,0.721582,0.474465,0.640857
1,6168,0.115321,0.100494,0.140033,0.423394,0.329489,0.545305,0.192751,0.141680,0.120264,...,0.128501,0.392092,0.336079,0.372323,0.286656,0.420099,0.802306,0.469522,0.158155,0.398682
2,8575,0.514003,0.790774,0.932455,0.037891,0.070840,0.039539,0.772652,0.889621,0.836903,...,0.617792,0.184514,0.349259,0.163097,0.472817,0.054366,0.095552,0.276771,0.701812,0.336079
3,618,0.456343,0.423394,0.370675,0.635914,0.535420,0.433278,0.359143,0.113674,0.922570,...,0.555189,0.654036,0.296540,0.474465,0.207578,0.375618,0.565074,0.393740,0.365733,0.319605
4,3471,0.509061,0.443163,0.347611,0.308072,0.846787,0.601318,0.397035,0.474465,0.469522,...,0.428336,0.049423,0.032949,0.019769,0.024712,0.087315,0.680395,0.940692,0.151565,0.009885
5,6343,0.507414,0.115321,0.046129,0.879736,0.775947,0.728171,0.108731,0.168040,0.196046,...,0.196046,0.838550,0.566722,0.532125,0.701812,0.777595,0.634267,0.771005,0.403624,0.835255
6,6987,0.296540,0.174629,0.464580,0.658979,0.482702,0.828666,0.472817,0.253707,0.261944,...,0.115321,0.023064,0.014827,0.034596,0.019769,0.031301,0.789127,0.803954,0.095552,0.013180
7,5871,0.624382,0.502471,0.337727,0.420099,0.774300,0.937397,0.532125,0.545305,0.283361,...,0.515651,0.037891,0.041186,0.051071,0.016474,0.042834,0.950577,0.991763,0.028007,0.001647
8,7842,0.537068,0.439868,0.388797,0.372323,0.891269,0.747941,0.293245,0.451400,0.538715,...,0.650741,0.874794,0.678748,0.652389,0.887974,0.686985,0.429984,0.423394,0.624382,0.795717
9,1589,0.461285,0.372323,0.428336,0.522241,0.594728,0.861615,0.528830,0.375618,0.225700,...,0.377265,0.579901,0.416804,0.660626,0.652389,0.591433,0.838550,0.818781,0.309720,0.502471
